<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/t5_base_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate evaluate rouge_score sentencepiece -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

from rouge_score import rouge_scorer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/t5_base_train.csv"

val_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/t5_base_validation.csv"

test_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/t5_base_test.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(42373, 2)
(5297, 2)
(5297, 2)


In [ ]:
train_df['model_input'] = train_df['model_input'].astype(str).str[:300]

val_df['model_input'] = val_df['model_input'].astype(str).str[:300]

test_df['model_input'] = test_df['model_input'].astype(str).str[:300]

In [ ]:
def format_text(row):

    return row['model_input'] + row['model_target']

train_texts = train_df.apply(format_text, axis=1)

val_texts = val_df.apply(format_text, axis=1)

test_texts = test_df.apply(format_text, axis=1)

print(train_texts.iloc[0])

summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asset-backed securities. Didi, which said in December it had raised $4 billion to support its overseas expansion, did not respond to a Reuters' requChina's Didi looks to raise $1.6 billion via asset-backed securities


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    pipeline
)

In [ ]:
model_name = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

torch.backends.cuda.matmul.allow_tf32 = True

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist()
})

val_dataset = Dataset.from_dict({
    "text": val_texts.tolist()
})

test_dataset = Dataset.from_dict({
    "text": test_texts.tolist()
})

In [ ]:
MAX_LENGTH = 48
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/42373 [00:00<?, ? examples/s]

Map:   0%|          | 0/5297 [00:00<?, ? examples/s]

Map:   0%|          | 0/5297 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

In [ ]:
training_args = TrainingArguments(

    output_dir="./headline_model",

    num_train_epochs=2,

    per_device_train_batch_size=32,

    per_device_eval_batch_size=32,

    learning_rate=5e-5,

    weight_decay=0.01,

    dataloader_num_workers=2,

    logging_steps=50,

    save_steps=100000,

    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator
)

In [ ]:
trainer.train()

Step,Training Loss
50,0.236306
100,0.007431
150,0.002787
200,0.001851
250,0.001989
300,0.001930
350,0.001250
400,0.001222
450,0.001468
500,0.000904


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2650, training_loss=0.005235296883181018, metrics={'train_runtime': 2031.9965, 'train_samples_per_second': 41.706, 'train_steps_per_second': 1.304, 'total_flos': 4838132380631040.0, 'train_loss': 0.005235296883181018, 'epoch': 2.0})

In [ ]:
torch.cuda.empty_cache()

print(
    "Allocated:",
    round(
        torch.cuda.memory_allocated()/1024**3,
        2
    ),
    "GB"
)

Allocated: 2.51 GB


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Current device:", torch.cuda.current_device())

print("Model device:", next(model.parameters()).device)

CUDA available: True
GPU: Tesla T4
Current device: 0
Model device: cuda:0


In [ ]:
save_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/t5_base_model"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!


In [ ]:
generator = pipeline(

    "text-generation",

    model=save_path,

    tokenizer=save_path,

    device=0
)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [ ]:
def generate_headline(article):

    article = str(article)[:500]

    prompt = f"Article: {article} Headline:"

    result = generator(

        prompt,

        max_new_tokens=12,

        num_beams=2,

        do_sample=False
    )

    text=result[0]["generated_text"]

    headline=text.split(
        "Headline:"
    )[-1].strip()

    return headline

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "/content/drive/MyDrive/headlineGenerationProjectNLP/t5_base_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

model.to(device)

model.eval()

print("✅ T5 model loaded successfully")


custom_articles = [

    """
    Scientists discovered a new Earth-like planet that may contain water
    and could potentially support life according to recent observations
    collected by NASA telescopes.
    """,

    """
    Heavy rainfall caused severe flooding across several cities in China,
    forcing thousands of residents to evacuate while emergency rescue teams
    continued operations throughout the region.
    """
]

def generate_headline(article):

    prompt = f"""
    Generate a short news headline for this article:

    {article}

    Headline:
    """

    inputs = tokenizer(

        prompt,

        return_tensors="pt",

        truncation=True,

        max_length=256

    ).to(device)

    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=12,

            min_new_tokens=5,

            num_beams=15,

            temperature=0.7,

            do_sample=True,

            top_k=50,

            top_p=0.95,

            repetition_penalty=4.0,

            no_repeat_ngram_size=4,

            length_penalty=3.0,

            early_stopping=True
        )

    headline = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True
    )

    headline = headline.replace("Headline:", "")
    headline = headline.replace("generate headline:", "")
    headline = headline.replace("Generate a short news headline for this article:", "")
    headline = headline.strip()

    return headline

print("\n===================================")
print("NEW HEADLINE RESULTS")
print("===================================\n")

for article in custom_articles:

    headline = generate_headline(article)

    print("ARTICLE:")
    print(article.strip())

    print("\nGENERATED HEADLINE:")
    print(headline)

    print("\n" + "="*70 + "\n")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ T5 model loaded successfully

NEW HEADLINE RESULTS

ARTICLE:
Scientists discovered a new Earth-like planet that may contain water
    and could potentially support life according to recent observations
    collected by NASA telescopes.

GENERATED HEADLINE:
: Scientists discovered a new Earth-like planet


ARTICLE:
Heavy rainfall caused severe flooding across several cities in China,
    forcing thousands of residents to evacuate while emergency rescue teams
    continued operations throughout the region.

GENERATED HEADLINE:
: Heavy rainfall caused severe flooding across several cities in China




In [ ]:
MODEL_PATH = "/content/drive/MyDrive/headlineGenerationProjectNLP/t5_base_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

print("✅ T5 model loaded successfully")

DATA_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"
test_df = pd.read_csv(
    os.path.join(DATA_DIR, "t5_base_test.csv")
)[["model_input", "model_target"]].dropna()

print("Test Samples:", len(test_df))

sample_size = min(200, len(test_df))
test_samples = test_df.sample(sample_size, random_state=42).reset_index(drop=True)

inputs = [f"summarize: {text}" for text in test_samples["model_input"].tolist()]
targets = test_samples["model_target"].tolist()

predictions = []
batch_size = 8

for i in range(0, len(inputs), batch_size):
    batch_texts = inputs[i:i + batch_size]

    encodings = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=encodings["input_ids"],
            attention_mask=encodings["attention_mask"],
            max_length=64,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    decoded_outputs = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )
    predictions.extend(decoded_outputs)

predictions = [p.strip() for p in predictions]
targets = [t.strip() for t in targets]

rouge = evaluate.load("rouge")
rouge_results = rouge.compute(predictions=predictions, references=targets)

print("\n===================================")
print("ROUGE RESULTS - T5 BASE")
print("===================================\n")
for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")

# Exact match
exact_matches = [pred.lower() == target.lower() for pred, target in zip(predictions, targets)]
exact_match_rate = np.mean(exact_matches)

print("\n===================================")
print(f"Exact Match Rate: {exact_match_rate:.4f}")
print("===================================\n")



Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ T5 model loaded successfully
Test Samples: 5297

ROUGE RESULTS

rouge1: 0.3501
rouge2: 0.2260
rougeL: 0.3284
rougeLsum: 0.3372

Exact Match Rate: 0.0353

